# Chocolate Sales — Boxes Shipped Prediction

Predict `Boxes_Shipped` for a single order from its pre-order features.

**Key finding:** the original public notebook scored R2 ~0.9997 — that was **target leakage**. The `Amount` column equals `Boxes_Shipped * Price_per_Box * (1 - Discount/100)`, so it encodes the answer. This notebook removes that leakage and reports an honest model.

In [ ]:
import pandas as pd
import numpy as np

## 1. Load

In [ ]:
df = pd.read_csv("Chocolate_Sales.csv")
df.head(2)

In [ ]:
df.info()

## 2. Drop the ID column (no predictive value)

In [ ]:
df = df.drop(columns=["Order_ID"])

## 3. Missing values

In [ ]:
df.isnull().sum()[df.isnull().sum() > 0]

Numeric columns -> median (robust to skew). For a production pipeline these medians should be fit on the train split only; for this notebook we fill on the full frame for simplicity.

In [ ]:
for col in ['Discount_Pct', 'Price_per_Box', 'Marketing_Spend']:
    df[col] = df[col].fillna(df[col].median())

`Order_Date` -> parse to datetime, fill the few missing with the mode (most common date).

In [ ]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")
df["Order_Date"] = df["Order_Date"].fillna(df["Order_Date"].mode()[0])

In [ ]:
df.isnull().sum()[df.isnull().sum() > 0]

## 4. Clean the target (`Boxes_Shipped`)

`describe()` shows a min of -1642 boxes, which is impossible. These are corrupted rows. We drop them (also drops any NaN target). Under 3% of the data.

In [ ]:
df['Boxes_Shipped'].describe()

In [ ]:
(df['Boxes_Shipped'] < 0).sum()

In [ ]:
df = df[df['Boxes_Shipped'] >= 0].copy()
print('rows remaining:', len(df))

## 5. Prove the `Amount` leakage, then drop it

If `Amount` is derivable from the target, it cannot be used as a feature (and would not be known at prediction time anyway). We check `Amount` vs the implied formula.

In [ ]:
df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")
expected = df["Boxes_Shipped"] * df["Price_per_Box"] * (1 - df["Discount_Pct"]/100)
diff = df["Amount"] - expected
diff.describe()   # median ~0 -> Amount is derived from the target -> leakage

In [ ]:
df = df.drop(columns=["Amount"])

## 6. Feature engineering

Date parts (capture seasonality) and `Discounted_Price` (actual price paid — known before the order ships). Every engineered feature uses only pre-order information; nothing derived from `Boxes_Shipped`.

In [ ]:
df["Order_Year"] = df["Order_Date"].dt.year
df["Order_Month"] = df["Order_Date"].dt.month
df["Order_Quarter"] = df["Order_Date"].dt.quarter
df["Order_Day"] = df["Order_Date"].dt.day
df["Order_Weekday"] = df["Order_Date"].dt.weekday
df = df.drop(columns=["Order_Date"])

In [ ]:
df["Discounted_Price"] = df["Price_per_Box"] * (1 - df["Discount_Pct"]/100)

## 7. One-hot encode categoricals

In [ ]:
cat_cols = df.select_dtypes(include=["object"]).columns.to_list()
df = pd.get_dummies(data=df, columns=cat_cols, drop_first=True, dtype="int")
print(df.shape)
df.head(2)

## 8. Features / target

In [ ]:
X = df.drop(columns=["Boxes_Shipped"])
y = df["Boxes_Shipped"]
print("target skew:", y.skew())   # high skew -> log1p helps

## 9. Train & evaluate

Target is right-skewed (skew ~3.5), so we train on `log1p(y)` and convert predictions back with `expm1` before scoring, so MAE/RMSE are in real box units.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_log = np.log1p(y)
X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

In [ ]:
def train_and_evaluate(name, model):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    y_true = np.expm1(y_test)
    y_pred = np.expm1(preds)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_test, preds)   # R2 on the log scale we trained on
    print(f'{name:15s}  MAE {mae:,.2f}   RMSE {rmse:,.2f}   R2 {r2:.3f}')
    return model

lr = train_and_evaluate('linear', LinearRegression())
rf = train_and_evaluate('random forest',
                        RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))

## 10. Feature importances

In [ ]:
pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)

## 11. Save the model bundle

`feature_order` stores the exact one-hot column layout — the API reindexes incoming orders against it. `log_target=True` reminds inference to `expm1` predictions.

In [ ]:
import pickle
bundle = {
    'model': rf,
    'feature_order': list(X.columns),
    'log_target': True,
}
with open('chocolate_model.pkl', 'wb') as f:
    pickle.dump(bundle, f)
print('saved chocolate_model.pkl')